In [52]:
import os
from typing import TypedDict

import anthropic
import psycopg
from dotenv import find_dotenv, load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langchain_core.tools import tool

# override=True makes .env authoritative. Without it, load_dotenv() leaves any
# ANTHROPIC_API_KEY already present in the kernel's environment untouched, so a
# kernel started before .env was fixed keeps using the stale key forever.
ENV_PATH = find_dotenv(usecwd=True)
load_dotenv(ENV_PATH, override=True)
DATABASE_URL = os.environ["DATABASE_URL"]
CURRENT_CUSTOMER_ID = 101

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip().strip("\"'")
if not ANTHROPIC_API_KEY.startswith("sk-ant-") or len(ANTHROPIC_API_KEY) < 90:
    raise ValueError(
        f"ANTHROPIC_API_KEY is unusable: got {len(ANTHROPIC_API_KEY)} characters, "
        f"expected 100+ starting with 'sk-ant-'. Loaded from: {ENV_PATH or 'NO .env FOUND'}\n"
        "  - 0 characters  -> no ANTHROPIC_API_KEY line in .env, or .env was not found\n"
        "  - short value   -> the key is truncated; paste the FULL key onto ONE line, no quotes, no line breaks\n"
        "Get a key at https://console.anthropic.com/settings/keys"
    )

llm = ChatAnthropic(
    model="claude-opus-5-5",
    api_key=ANTHROPIC_API_KEY,
    max_tokens=16000,
)
print("Model loaded successfully.")

Model loaded successfully.


In [53]:
def get_connection():
    return psycopg.connect(DATABASE_URL)

ORDER_COLUMNS = "order_id, restaurant, items, status, total_amount, created_at, promised_at, delivered_at"

def order_from_row(row):
    minutes_late = None
    if row[6] and row[7]:
        minutes_late = round((row[7] - row[6]).total_seconds() / 60)
    return {
        "order_id": row[0], "restaurant": row[1], "items": row[2], "status": row[3],
        "total_amount": float(row[4]), "created_at": str(row[5]),
        "promised_at": str(row[6]) if row[6] else None,
        "delivered_at": str(row[7]) if row[7] else None,
        "minutes_late": minutes_late,
    }

def query_latest_order(customer_id):
    with get_connection() as conn:
        return conn.execute(
            f"SELECT {ORDER_COLUMNS} FROM orders WHERE customer_id=%s ORDER BY created_at DESC LIMIT 1",
            (customer_id,),
        ).fetchone()

def query_order(customer_id, order_id):
    with get_connection() as conn:
        return conn.execute(
            f"SELECT {ORDER_COLUMNS} FROM orders WHERE order_id=%s AND customer_id=%s",
            (order_id, customer_id),
        ).fetchone()
print("Database connection established and initialized successfully.")        

Database connection established and initialized successfully.


In [54]:
from langchain_core.tools import tool

missing = [
    f"  - {name:<22} defined in {where}"
    for name, where in {"query_latest_order": "the database helpers cell", "query_order": "the database helpers cell", "order_from_row": "the database helpers cell"}.items()
    if name not in globals()
]
if missing:
    raise NameError(
        "Earlier cells have not run in this kernel:\\n" + "\\n".join(missing)
        + "\\n\\nUse Run All (or Run All Above) and then re-run this cell."
    )


@tool
def basic_get_latest_order() -> dict:
    """Get the latest order for the current classroom customer."""
    row = query_latest_order(CURRENT_CUSTOMER_ID)
    if row is None:
        return {"found": False, "message": "No orders were found."}
    return {"found": True, "order": order_from_row(row)}

@tool
def basic_get_order_details(order_id: str) -> dict:
    """Get details for an order ID belonging to the current classroom customer."""
    row = query_order(CURRENT_CUSTOMER_ID, order_id)
    if row is None:
        return {"found": False, "message": "No matching order was found."}
    return {"found": True, "order": order_from_row(row)}

In [55]:
print("Tools loaded successfully.")

Tools loaded successfully.


### LangChain Agent

In [56]:
from langchain.agents import create_agent

# Names this cell needs from earlier cells; report them all at once.
missing = [
    f"  - {name:<26} defined in {where}"
    for name, where in {
        "llm": "the first cell",
        "basic_get_latest_order": "the tools cell above",
        "basic_get_order_details": "the tools cell above",
    }.items()
    if name not in globals()
]
if missing:
    raise NameError(
        "Earlier cells have not run in this kernel:\n" + "\n".join(missing)
        + "\n\nUse Run All (or Run All Above) and then re-run this cell."
    )

BASIC_SYSTEM_PROMPT = """You are a Zomato customer-support assistant.
Use tools to verify order information. Never invent order details.
If a tool says an order was not found, explain that you could not verify it.
Be concise and do not claim to have performed actions you did not perform."""

basic_tools = [basic_get_latest_order, basic_get_order_details]

basic_agent = create_agent(
    model=llm,
    tools=basic_tools,
    system_prompt=BASIC_SYSTEM_PROMPT,
)
print("Agent tools:", [t.name for t in basic_tools])

Agent tools: ['basic_get_latest_order', 'basic_get_order_details']


In [57]:
import anthropic

missing = [
    f"  - {name:<22} defined in {where}"
    for name, where in {"basic_agent": "the agent cell above"}.items()
    if name not in globals()
]
if missing:
    raise NameError(
        "Earlier cells have not run in this kernel:\\n" + "\\n".join(missing)
        + "\\n\\nUse Run All (or Run All Above) and then re-run this cell."
    )

API_ERROR_HINTS = {
    400: "the request was rejected -- if it mentions 'credit balance', add credits at https://console.anthropic.com/settings/billing",
    401: "the key sent to Anthropic is wrong, truncated, or revoked -- check ANTHROPIC_API_KEY in .env",
    403: "the key is recognised but not permitted for this request -- wrong workspace/model access, or a disabled key",
    404: "the model name is unknown to this account",
    429: "rate limited -- slow down and retry",
}


def ask_basic(question):
    try:
        result = basic_agent.invoke({
            "messages": [{"role": "user", "content": question}]
        })
    except anthropic.APIStatusError as error:
        hint = API_ERROR_HINTS.get(error.status_code, "see https://docs.anthropic.com/en/api/errors")
        print(f"ANTHROPIC API ERROR {error.status_code}: {hint}")
        print("Detail:", error.message)
        if error.status_code in (401, 403):
            print("After editing .env, restart the kernel -- load_dotenv() will not refresh an already-loaded value.")
        return None

    for message in result["messages"]:
        if getattr(message, "tool_calls", None):
            print("MODEL REQUESTED:", [(call["name"], call["args"]) for call in message.tool_calls])
        elif message.type == "tool":
            print("TOOL RESULT:", str(message.content)[:400])
    print("FINAL:", result["messages"][-1].content)
    return result


ask_basic("Where is my latest order?")

MODEL REQUESTED: [('basic_get_latest_order', {})]
TOOL RESULT: {"found": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status": "OUT_FOR_DELIVERY", "total_amount": 350.0, "created_at": "2026-11-06 20:14:37.166546+00:00", "promised_at": "2026-11-06 20:54:37.166546+00:00", "delivered_at": null, "minutes_late": null}}
FINAL: [{'signature': 'CAQSsQUKEAgTGAI4AUIIdGhpbmtpbmcSDA+udq87oJifnZePzBoMVO8TxPC/4A2BZ7iHIjB8w0ng7gOkmTHP0QXIBx5F6pMW4cvlCl9FtSf5LIwABE3i9jIsTpys8YJK0/LUssMqzgSPz+uTCJ5oq+GYsUt5/YER4rLT/p8UaLmAGAAb25Pmqz852iToDyvm0YwLV9wcKpB9uqJq9K2hoRQNk1zUCONHLPs1u+O9RI8U+tmaMPvOa9uQSEC81FzMgI+1eWzn/ttddJFCMhO5Xh5reG5zKU9RjIBbWBg6faZDZiqRIiFdM8D1C4fn2AeQYIM+bORNFPd1Jp+eWPvOZ00dySr+Bjnxr/hM28M0/tlvSTrdUMScfrDokMpGGRyxVxKggF5ybPFAGIE+XSj671lIz2f3PVdl1kDr9dY7b7i1rCh4CLrhuztdrffQqpUr+mVSP+H8wIfKH6vACYAVFj7YxikN7YQ/5DxSu0yO6AE5wHJ3TAKZt5HEbvhIj47nmrHO4jl9pYv7rg/h1voWHAPp3fKl3Oi25NKHIcccgOxKEcuY+ed5NpCrS/+i8Jrs9nHX4RxE61JFRPpQkOyQ7L7q

{'messages': [HumanMessage(content='Where is my latest order?', additional_kwargs={}, response_metadata={}, id='3d8d636a-dcc3-45c8-b749-89035f1ef0a7'),
  AIMessage(content=[{'id': 'toolu_01AaiTADzzedsRYbtgfHoFK9', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'basic_get_latest_order', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011CgnzzxFpSswGET91c6dMb', 'container': None, 'diagnostics': None, 'model': 'claude-opus-5-5', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 533, 'output_tokens': 32, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard', 'speed': 'standard'}, 'model_name': 'claude-opus-5-5', 'model_provider': 'anthropic'}, id='lc_run--01a1b470-775c-71a4

In [58]:
missing = [
    f"  - {name:<22} defined in {where}"
    for name, where in {"ask_basic": "the cell that defines ask_basic"}.items()
    if name not in globals()
]
if missing:
    raise NameError(
        "Earlier cells have not run in this kernel:\\n" + "\\n".join(missing)
        + "\\n\\nUse Run All (or Run All Above) and then re-run this cell."
    )

ask_basic("Show me the details for ORD-1001.")

MODEL REQUESTED: [('basic_get_order_details', {'order_id': 'ORD-1001'})]
TOOL RESULT: {"found": true, "order": {"order_id": "ORD-1001", "restaurant": "Biryani Blues", "items": "Chicken Biryani x1; Raita x1", "status": "DELIVERED", "total_amount": 480.0, "created_at": "2026-11-05 15:14:37.166546+00:00", "promised_at": "2026-11-05 15:59:37.166546+00:00", "delivered_at": "2026-11-05 16:39:37.166546+00:00", "minutes_late": 40}}
FINAL: Here are the details for order **ORD-1001**:

- **Restaurant:** Biryani Blues
- **Items:** Chicken Biryani x1, Raita x1
- **Total:** ₹480.00
- **Status:** Delivered
- **Ordered at:** 5 Nov 2026, 15:14 UTC
- **Promised by:** 5 Nov 2026, 15:59 UTC
- **Delivered at:** 5 Nov 2026, 16:39 UTC

Your order arrived **40 minutes late**. I'm sorry about that. If you'd like to raise a concern about the delay, I can help you with next steps.


{'messages': [HumanMessage(content='Show me the details for ORD-1001.', additional_kwargs={}, response_metadata={}, id='95f8ebda-a6ce-484a-9870-743b294611ea'),
  AIMessage(content=[{'id': 'toolu_01LX3XDM68HqpJdBbvcFRtC7', 'caller': {'type': 'direct'}, 'input': {'order_id': 'ORD-1001'}, 'name': 'basic_get_order_details', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011Cgo11BhTGnCobtfvB1mWC', 'container': None, 'diagnostics': None, 'model': 'claude-opus-5-5', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 538, 'output_tokens': 59, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard', 'speed': 'standard'}, 'model_name': 'claude-opus-5-5', 'model_provider': 'anthropic'},

In [59]:
missing = [
    f"  - {name:<22} defined in {where}"
    for name, where in {"ask_basic": "the cell that defines ask_basic"}.items()
    if name not in globals()
]
if missing:
    raise NameError(
        "Earlier cells have not run in this kernel:\\n" + "\\n".join(missing)
        + "\\n\\nUse Run All (or Run All Above) and then re-run this cell."
    )

ask_basic("Show me details for ORD-1001.")

MODEL REQUESTED: [('basic_get_order_details', {'order_id': 'ORD-1001'})]
TOOL RESULT: {"found": true, "order": {"order_id": "ORD-1001", "restaurant": "Biryani Blues", "items": "Chicken Biryani x1; Raita x1", "status": "DELIVERED", "total_amount": 480.0, "created_at": "2026-11-05 15:14:37.166546+00:00", "promised_at": "2026-11-05 15:59:37.166546+00:00", "delivered_at": "2026-11-05 16:39:37.166546+00:00", "minutes_late": 40}}
FINAL: [{'signature': 'CAQSoQUKEAgTGAI4AUIIdGhpbmtpbmcSDJlVcp8wnuLonmAhQxoMPr0dg6/NGS3GH2dJIjCK4zTGyfc09Z48mtsfntnTtCghY40Cbd/ywUK/PavQw+xav/fywLZMAVYwOomBItEqvgSVqogjK3wDQX6obCQd4fAzvIjqyNLAylhisp8qW4MoQmiC3X1JxnoLKQLP24HUYzqD3awC50GSggVxW5IpTEOBY8l2uks75Y61c8+EHvk5m8CnKD5Vv7g5Acp2fvrl+LaY4uYq7+GtdVzcPPhxsbp69zBON1WE6DzoPgfcvYyNRgdTDIiAknB1FvSeBn9CTtjn1ryYd4gA51WHGRU1t/vCxyDZaYD6sZIJ0pK8S92tWIEK1F6bLp1aUKrDkThFvreELNGf5e6pOmf2eELCYKJRBQl5gNbunCwfrF48GIe48/+HPTtdq5m+G1dK1AtE98OJ+l7TCY8HC5PWBb1PkM/ytnX71ntpOlIL5zywnU9kBpp5WnxoeuOT3RJdp4s1pxJ4Avta4zUtZmcWDvDmhHK8ZMq7u

{'messages': [HumanMessage(content='Show me details for ORD-1001.', additional_kwargs={}, response_metadata={}, id='a117cb80-e800-455b-89a8-4dcb44594fd0'),
  AIMessage(content=[{'id': 'toolu_01XCJhkZfX939qfs2orT9vtF', 'caller': {'type': 'direct'}, 'input': {'order_id': 'ORD-1001'}, 'name': 'basic_get_order_details', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011Cgo11QPSk3or9A7dJeXbd', 'container': None, 'diagnostics': None, 'model': 'claude-opus-5-5', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 537, 'output_tokens': 59, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard', 'speed': 'standard'}, 'model_name': 'claude-opus-5-5', 'model_provider': 'anthropic'}, id=

#### Harden The Langchain Agent

##### 1. Validate withPydantic

In [60]:
from pydantic import BaseModel, Field

class OrderInput(BaseModel):
    order_id: str = Field(min_length=1, max_length=40, pattern=r"^ORD-\d+$",
                          description="Order ID such as ORD-1001",
)

class RefundInput(BaseModel):
    order_id: str = Field(min_length=1, max_length=40, pattern=r"^ORD-[0-9]+$")
    amount: float = Field(gt=0, le=10000)
    reason: str = Field(min_length=5, max_length=300, description="Reason for refund request")


##### 2. Validate Identity

In [61]:
from typing import TypedDict

import psycopg
from langchain_core.tools import tool

missing = [
    f"  - {name:<22} defined in {where}"
    for name, where in {"OrderInput": "'1. Validate with Pydantic'", "query_latest_order": "the database helpers cell", "query_order": "the database helpers cell", "order_from_row": "the database helpers cell"}.items()
    if name not in globals()
]
if missing:
    raise NameError(
        "Earlier cells have not run in this kernel:\\n" + "\\n".join(missing)
        + "\\n\\nUse Run All (or Run All Above) and then re-run this cell."
    )

from typing import TypedDict

from langchain.tools import ToolRuntime

class AgentContext(TypedDict):
    customer_id: int

@tool(args_schema=OrderInput)
def get_order_details(order_id: str, runtime: ToolRuntime[AgentContext]) -> dict:
    """Get details of an order belonging to the current verified customer."""
    customer_id = runtime.context["customer_id"]
    try:
        row = query_order(customer_id, order_id)
    except psycopg.Error as exc:
        print("[log] get_order_details failed:", type(exc).__name__)
        return {"ok": False, "code": "SERVICE_UNAVAILABLE", "message": "I could not look up that order right now."}
    if row is None:
        return {"ok": False, "code": "ORDER_NOT_FOUND", "message": "No matching order was found."}
    return {"ok": True, "order": order_from_row(row)}


@tool
def get_latest_order(runtime: ToolRuntime[AgentContext]) -> dict:
    """Get the latest order belonging to the current verified customer."""
    customer_id = runtime.context["customer_id"]
    try:
        row = query_latest_order(customer_id)
    except psycopg.Error as exc:
        print("[log] get_latest_order failed:", type(exc).__name__)
        return {"ok": False, "code": "SERVICE_UNAVAILABLE", "message": "I could not look up your latest order right now."}
    if row is None:
        return {"ok": False, "code": "ORDER_NOT_FOUND", "message": "No orders were found."}
    return {"ok": True, "order": order_from_row(row)}

In [62]:
from langchain.agents import create_agent

# Names this cell needs from earlier cells; report them all at once.
missing = [
    f"  - {name:<22} defined in {where}"
    for name, where in {
        "llm": "the first cell",
        "BASIC_SYSTEM_PROMPT": "the 'LangChain Agent' cell above",
        "AgentContext": "'2. Validate Identity'",
        "get_latest_order": "'2. Validate Identity' (or '4. Safe Errors')",
        "get_order_details": "'2. Validate Identity' (or '4. Safe Errors')",
    }.items()
    if name not in globals()
]
if missing:
    raise NameError(
        "Earlier cells have not run in this kernel:\n" + "\n".join(missing)
        + "\n\nUse Run All (or Run All Above) and then re-run this cell."
    )

context_tools = [get_latest_order, get_order_details]
context_agent = create_agent(
    model=llm,
    tools=context_tools,
    system_prompt=BASIC_SYSTEM_PROMPT,
    context_schema=AgentContext,
)

def ask_as_customer(question, customer_id=101):
    result = context_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        context={"customer_id": customer_id},
    )
    print(result["messages"][-1].content)
    return result

ask_as_customer("Show me details for ORDER-2001.", customer_id=101)

[{'signature': 'CAQS3AcKEAgTGAI4AUIIdGhpbmtpbmcSDNbUUbuBlgWT5J1VOxoMRNLsZTDkCsJxOPEfIjANcNT374kDhM6sdpaG0qoHpH512ftqTGg6kRPdCRxwouGKwD3QDcsQ56mKlV2KLNkq+QbykFNh0ZhuryoCq1rwUXOHUE8QYaqXLXWf6tnsohAPNKBvJNzC1khTKH1P/Q/mCkvZL9Pa2XSt6tJth/Pig9670l2N0ZW0u76Q+5dJTdwVkaTfCj0x9XAsL5/1cL3HQRr/F6UI9McpQMQuxxRiryjOZA8D2bT0tGwYMm4kS5wJ/83npUoavWWxYcbxLFgv/ugFQmtczfWl1ghEK3IH3b90wexxEYl+dQ+U+QHlOjvQJgMHz7LeFbFuPTLQKGeYi7uE6L7ACR3Sll+3Qs7t2ekLKk7qVpiAQ5cejqmy9jwV0vEzslJ6caqDo0cqbdD9ybIJFz1+lcz/Wh/lauExVRXqBpCssZjxaJdTUOOO/85wkrmp+OHGtcQ+wfa8lO0buICU6IiskZ5JQuzO+hYDgjTXOpxi6kb6BxulGqheMqK18dj1qnkHgoVoH96TEADgypHqgklLt3qI9iwF37scxuw/yNWrd/olywaN1JGu+lEYgVnBfvWSxrSnQEALlGrT7hTmKue8t7u326W1nDaSH54dNBXXZRohB4hEovJ/uOktrBnILxFdxHXekc5XH2MI7SoUsOZO7ZI/0dHabBrDUmpoFogBr4nDNpxG8T9FtjEwSYgdpPKaZ/eDhnVmN1RH6JoYWueRQNPN61F8dVGk4tDczoYLbLMx/S4MSbF7N7zdWY4h+8yBckcyADWgHouQaEQ1U8mOqsvy7w/ttuEFk+tuI0sAS0i72qMtQmQ6oVse7y3wjoRKjs8fO2vAEo9062WuCLoPd4q2tm7+ykw6NIQvibHhGI5Ugjzh6bfbywmlPDJSLUm6ctnviUI7dnCYZy5X9UkadkJRQy9B

{'messages': [HumanMessage(content='Show me details for ORDER-2001.', additional_kwargs={}, response_metadata={}, id='3e4566cd-ec47-4e79-b4f2-f91620386968'),
  AIMessage(content=[{'signature': 'CAQS3AcKEAgTGAI4AUIIdGhpbmtpbmcSDNbUUbuBlgWT5J1VOxoMRNLsZTDkCsJxOPEfIjANcNT374kDhM6sdpaG0qoHpH512ftqTGg6kRPdCRxwouGKwD3QDcsQ56mKlV2KLNkq+QbykFNh0ZhuryoCq1rwUXOHUE8QYaqXLXWf6tnsohAPNKBvJNzC1khTKH1P/Q/mCkvZL9Pa2XSt6tJth/Pig9670l2N0ZW0u76Q+5dJTdwVkaTfCj0x9XAsL5/1cL3HQRr/F6UI9McpQMQuxxRiryjOZA8D2bT0tGwYMm4kS5wJ/83npUoavWWxYcbxLFgv/ugFQmtczfWl1ghEK3IH3b90wexxEYl+dQ+U+QHlOjvQJgMHz7LeFbFuPTLQKGeYi7uE6L7ACR3Sll+3Qs7t2ekLKk7qVpiAQ5cejqmy9jwV0vEzslJ6caqDo0cqbdD9ybIJFz1+lcz/Wh/lauExVRXqBpCssZjxaJdTUOOO/85wkrmp+OHGtcQ+wfa8lO0buICU6IiskZ5JQuzO+hYDgjTXOpxi6kb6BxulGqheMqK18dj1qnkHgoVoH96TEADgypHqgklLt3qI9iwF37scxuw/yNWrd/olywaN1JGu+lEYgVnBfvWSxrSnQEALlGrT7hTmKue8t7u326W1nDaSH54dNBXXZRohB4hEovJ/uOktrBnILxFdxHXekc5XH2MI7SoUsOZO7ZI/0dHabBrDUmpoFogBr4nDNpxG8T9FtjEwSYgdpPKaZ/eDhnVmN1RH6JoYWueRQNPN61F8dVGk4tDczoYLbL

##### 3. Least Priviledge

In [63]:
# Self-contained on purpose: this cell re-reads .env and defines its own
# connection strings, so it runs on a fresh kernel without the cells above.
import os

import psycopg
from dotenv import find_dotenv, load_dotenv

ENV_PATH = find_dotenv(usecwd=True)
load_dotenv(ENV_PATH, override=True)


def require_env(name, hint):
    value = os.environ.get(name, "").strip().strip("\"'")
    if not value:
        raise ValueError(
            f"{name} is missing. Loaded .env from: {ENV_PATH or 'NO .env FOUND'}\n  {hint}"
        )
    return value


DATABASE_URL = require_env(
    "DATABASE_URL",
    "This is the owner connection used to apply the grants below.",
)
READ_DATABASE_URL = require_env(
    "READ_DATABASE_URL",
    "Add to .env: READ_DATABASE_URL=postgresql://support_agent_ro:<password>@localhost:5432/zomato",
)

# Only privileges here. The refund table is created and granted in
# '5. Idempotent Writes', which owns that schema.
SETUP_SQL = """
GRANT CONNECT ON DATABASE zomato TO support_agent_ro, support_agent_rw;
GRANT USAGE ON SCHEMA public TO support_agent_ro, support_agent_rw;

-- Support agent: read orders, nothing else.
GRANT SELECT ON public.orders TO support_agent_ro;
REVOKE INSERT, UPDATE, DELETE ON public.orders FROM support_agent_ro;

-- The refund agent gets no access to orders at all.
REVOKE ALL ON public.orders FROM support_agent_rw;
"""

with psycopg.connect(DATABASE_URL, connect_timeout=5) as conn:
    conn.execute(SETUP_SQL)

# Read the resulting privileges straight from the catalog -- no guessing.
with psycopg.connect(DATABASE_URL, connect_timeout=5) as conn:
    tables = [
        name
        for name in ("orders", "refund_requests")
        if conn.execute("SELECT to_regclass(%s) IS NOT NULL", (f"public.{name}",)).fetchone()[0]
    ]
    for role in ("support_agent_ro", "support_agent_rw"):
        for table in tables:
            granted = [
                privilege
                for privilege in ("SELECT", "INSERT", "UPDATE", "DELETE")
                if conn.execute(
                    "SELECT has_table_privilege(%s, %s, %s)",
                    (role, f"public.{table}", privilege),
                ).fetchone()[0]
            ]
            print(f"  {role:17s} {table:16s} -> {', '.join(granted) or 'no access'}")
    if "refund_requests" not in tables:
        print("  (refund_requests not created yet -- see '5. Idempotent Writes')")

# Reading orders as the support role is allowed.
with psycopg.connect(READ_DATABASE_URL, connect_timeout=5) as conn:
    count = conn.execute("SELECT COUNT(*) FROM public.orders").fetchone()[0]
    print(f"\nSELECT succeeded: {count} orders visible to support_agent_ro")

# Deleting is not. Caught, so the demonstration does not abort the notebook: a
# prompt injection that talks the agent into deleting rows dies in the database,
# whatever the model was persuaded to attempt.
try:
    with psycopg.connect(READ_DATABASE_URL, connect_timeout=5) as conn:
        conn.execute("DELETE FROM public.orders WHERE order_id = %s", ("ORD-1001",))
except psycopg.errors.InsufficientPrivilege as exc:
    print("DELETE correctly denied:", str(exc).strip().splitlines()[0])
else:
    raise AssertionError("DELETE succeeded -- the read-only role is over-privileged!")

with psycopg.connect(DATABASE_URL, connect_timeout=5) as conn:
    print("Orders still present:", conn.execute("SELECT COUNT(*) FROM public.orders").fetchone()[0])

  support_agent_ro  orders           -> SELECT
  support_agent_ro  refund_requests  -> no access
  support_agent_rw  orders           -> no access
  support_agent_rw  refund_requests  -> SELECT, INSERT

SELECT succeeded: 6 orders visible to support_agent_ro
DELETE correctly denied: permission denied for table orders
Orders still present: 6


##### 4. Safe Errors

In [64]:
import os
import time

import psycopg
from dotenv import find_dotenv, load_dotenv
from langchain.tools import ToolRuntime
from langchain_core.tools import tool

load_dotenv(find_dotenv(usecwd=True), override=True)
DATABASE_URL = os.environ["DATABASE_URL"]

# This cell rebuilds the hardened tools on top of objects defined earlier, so
# report every missing one at once instead of one NameError per re-run.
REQUIRED_NAMES = {
    "OrderInput": "'1. Validate with Pydantic'",
    "AgentContext": "'2. Validate Identity'",
    "order_from_row": "the database helpers cell near the top",
    "query_order": "the database helpers cell near the top",
    "query_latest_order": "the database helpers cell near the top",
}
missing = [f"  - {name:<20} defined in {where}" for name, where in REQUIRED_NAMES.items() if name not in globals()]
if missing:
    raise NameError(
        "This cell builds on earlier cells that have not run in this kernel:\n"
        + "\n".join(missing)
        + "\n\nUse Run All (or Run All Above) and then re-run this cell."
    )

def get_connection():
    return psycopg.connect(
        DATABASE_URL,
        connect_timeout=5,
        options="-c statement_timeout=5000",
    )

def run_read_with_retry(query_fn, *args, attempts=2):
    for attempt in range(attempts):
        try:
            return query_fn(*args)
        except (psycopg.OperationalError, psycopg.InterfaceError):
            if attempt == attempts - 1:
                raise
            time.sleep(0.2 * (attempt + 1))

# Rebuild the tools so the hardened version uses bounded retries for reads.
@tool(args_schema=OrderInput)
def get_order_details(order_id: str, runtime: ToolRuntime[AgentContext]) -> dict:
    """Get details of an order belonging to the current verified customer."""
    customer_id = runtime.context["customer_id"]
    try:
        row = run_read_with_retry(query_order, customer_id, order_id)
    except psycopg.Error as exc:
        print("[log] get_order_details failed:", type(exc).__name__)
        return {"ok": False, "code": "SERVICE_UNAVAILABLE", "message": "I could not look up that order right now."}
    if row is None:
        return {"ok": False, "code": "ORDER_NOT_FOUND", "message": "No matching order was found."}
    return {"ok": True, "order": order_from_row(row)}

@tool
def get_latest_order(runtime: ToolRuntime[AgentContext]) -> dict:
    """Get the latest order belonging to the current verified customer."""
    customer_id = runtime.context["customer_id"]
    try:
        row = run_read_with_retry(query_latest_order, customer_id)
    except psycopg.Error as exc:
        print("[log] get_latest_order failed:", type(exc).__name__)
        return {"ok": False, "code": "SERVICE_UNAVAILABLE", "message": "I could not look up your latest order right now."}
    if row is None:
        return {"ok": False, "code": "ORDER_NOT_FOUND", "message": "No orders were found."}
    return {"ok": True, "order": order_from_row(row)}

print("Read tools now use statement/connect timeouts and bounded retries.")

Read tools now use statement/connect timeouts and bounded retries.


##### 5. Idempotent Writes

In [65]:
def ok(**data):
    return {"ok": True, **data}

def fail(code, message, **data):
    return {"ok": False, "code": code, "message": message, **data}

In [66]:
import os

import psycopg
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=True)
DATABASE_URL = os.environ["DATABASE_URL"]

missing = [
    f"  - {name:<18} defined in {where}"
    for name, where in {
        "get_connection": "the database helpers cell",
        "ok": "the ok/fail helpers cell above",
        "fail": "the ok/fail helpers cell above",
    }.items()
    if name not in globals()
]
if missing:
    raise NameError(
        "Earlier cells have not run in this kernel:\n" + "\n".join(missing)
        + "\n\nUse Run All (or Run All Above) and then re-run this cell."
    )

create_refunds_sql = """
CREATE TABLE IF NOT EXISTS refund_requests (
    id BIGSERIAL PRIMARY KEY,
    order_id TEXT NOT NULL REFERENCES orders(order_id),
    customer_id BIGINT NOT NULL REFERENCES customers(customer_id),
    amount NUMERIC(10,2) NOT NULL CHECK (amount > 0),
    reason TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'PENDING' CHECK (status IN ('PENDING','APPROVED','REJECTED')),
    idempotency_key TEXT NOT NULL UNIQUE,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
"""
# The refund agent may append a request and read it back, never rewrite one.
# Granted here, beside the schema, so the two cannot drift apart.
refund_grants_sql = """
GRANT SELECT, INSERT ON public.refund_requests TO support_agent_rw;
REVOKE UPDATE, DELETE ON public.refund_requests FROM support_agent_rw;
GRANT USAGE, SELECT ON SEQUENCE public.refund_requests_id_seq TO support_agent_rw;
REVOKE ALL ON public.refund_requests FROM support_agent_ro;
"""

with psycopg.connect(DATABASE_URL, connect_timeout=5) as conn:
    conn.execute(create_refunds_sql)
    conn.execute(refund_grants_sql)
print("Refund request table is ready.")

def create_refund_request(customer_id, order_id, amount, reason):
    with get_connection() as conn:
        order = conn.execute(
            "SELECT status, total_amount FROM orders WHERE order_id=%s AND customer_id=%s",
            (order_id, customer_id),
        ).fetchone()
        if order is None:
            return fail("ORDER_NOT_FOUND", "No matching order was found.")
        if order[0] != "DELIVERED":
            return fail("ORDER_NOT_DELIVERED", "Only delivered orders can be considered for a refund request.")
        if amount <= 0 or amount > float(order[1]):
            return fail("INVALID_AMOUNT", "The requested amount is outside the allowed range.")

        key = f"refund:{customer_id}:{order_id}"
        row = conn.execute(
            """INSERT INTO refund_requests(order_id, customer_id, amount, reason, idempotency_key)
               VALUES (%s,%s,%s,%s,%s) ON CONFLICT(idempotency_key) DO NOTHING
               RETURNING id, status""",
            (order_id, customer_id, amount, reason, key),
        ).fetchone()
        if row:
            return ok(code="REFUND_REQUESTED", refund_request_id=row[0], status=row[1],
                      message="Request created for human review. No money has been refunded.")
        existing = conn.execute(
            "SELECT id, status FROM refund_requests WHERE idempotency_key=%s", (key,)
        ).fetchone()
        return fail("ALREADY_REQUESTED", "A request already exists for this order.",
                    refund_request_id=existing[0], status=existing[1])

print(create_refund_request(101, "ORD-1001", 50, "Delivered later than promised"))


Refund request table is ready.
{'ok': True, 'code': 'REFUND_REQUESTED', 'refund_request_id': 7, 'status': 'PENDING', 'message': 'Request created for human review. No money has been refunded.'}


##### 6. Retries

In [67]:
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv(usecwd=True), override=True)

# OpenRouter speaks the OpenAI wire protocol, so ChatOpenAI drives it -- only the
# base_url and key differ. Both must be set in .env; name them in one error
# rather than failing on whichever os.environ[...] is reached first.
missing = [name for name in ("OPENROUTER_MODEL", "OPENROUTER_API_KEY") if not os.environ.get(name, "").strip()]
if missing:
    raise ValueError(
        f"Missing from .env: {', '.join(missing)}\n"
        "  OPENROUTER_MODEL   e.g. nvidia/nemotron-3-ultra-550b-a55b:free\n"
        "  OPENROUTER_API_KEY from https://openrouter.ai/settings/keys"
    )

reliable_llm = ChatOpenAI(
    model=os.environ["OPENROUTER_MODEL"],
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
    timeout=30,
    max_retries=2,
)
print("Fallback model ready:", reliable_llm.model_name)

Fallback model ready: nvidia/nemotron-3.5-super-vl:free


In [68]:
@tool(args_schema=RefundInput)
def request_refund(order_id: str, amount: float, reason: str, runtime: ToolRuntime[AgentContext]) -> dict:
    """Create a pending refund request for the current customer's delivered order. Does not issue a refund."""
    customer_id = runtime.context["customer_id"]
    try:
        return create_refund_request(customer_id, order_id, amount, reason)
    except psycopg.Error as exc:
        print("[log] request_refund failed:", type(exc).__name__)
        return fail("SERVICE_UNAVAILABLE", "The refund system is unavailable right now.")

print("Refund tool arguments:", request_refund.args)

Refund tool arguments: {'order_id': {'maxLength': 40, 'minLength': 1, 'pattern': '^ORD-[0-9]+$', 'title': 'Order Id', 'type': 'string'}, 'amount': {'exclusiveMinimum': 0, 'maximum': 10000, 'title': 'Amount', 'type': 'number'}, 'reason': {'description': 'Reason for refund request', 'maxLength': 300, 'minLength': 5, 'title': 'Reason', 'type': 'string'}}


#### Middlewares

##### Execution Limits

In [69]:
SYSTEM_PROMPT = """You are a Zomato customer-support assistant.
- Use tools to verify order details; never invent them.
- Use get_order_details for a specific order and get_latest_order for the latest order.
- Only propose a refund when verified order details and business rules support it.
- request_refund creates a PENDING request for human review. It never issues a refund.
- If a tool reports a failure or insufficient evidence, explain that you could not verify the information.
- Be concise and never expose internal error details.
"""

support_tools = [get_latest_order, get_order_details, request_refund]

In [70]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware

limited_agent = create_agent(
    model=reliable_llm,
    tools=support_tools,
    system_prompt=SYSTEM_PROMPT,
    context_schema=AgentContext,
    middleware=[
        ModelCallLimitMiddleware(run_limit=6, exit_behavior="end"),
        ToolCallLimitMiddleware(run_limit=8, exit_behavior="end"),                          # all tools together
        ToolCallLimitMiddleware(tool_name="get_order_details", run_limit=3, exit_behavior="end"),  # one tool on its own
    ],
)

result = limited_agent.invoke(
    {"messages": [{"role": "user", "content": "Where is my latest order?"}]},
    context={"customer_id": 101},
)

print(result["messages"][-1].content)

Your latest order (ORD-1003) from **Burger Barn** is currently **out for delivery**. It’s expected by **8:54 PM** today.


##### 2. Retry And Fallback Middleware

In [71]:
from langchain.agents.middleware import ModelFallbackMiddleware, ModelRetryMiddleware, ToolRetryMiddleware

backup_llm = ChatOpenAI(
    model=os.environ.get("OPENROUTER_FALLBACK_MODEL", os.environ["OPENROUTER_MODEL"]),
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
    timeout=30,
)

resilient_agent = create_agent(
    model=reliable_llm,
    tools=support_tools,
    system_prompt=SYSTEM_PROMPT,
    context_schema=AgentContext,
    middleware=[
        ModelFallbackMiddleware(backup_llm),                       # outer: tried only if the retries below fail
        ModelRetryMiddleware(max_retries=2, on_failure="error"),   # inner: retry the primary model first
        ToolRetryMiddleware(
            max_retries=2,
            tools=["get_order_details", "get_latest_order"],       # reads only, never request_refund
            retry_on=(psycopg.OperationalError, TimeoutError),     # transient failures only
            on_failure="error",
        ),
    ],
)
print("Built an agent with model retry + fallback and read-only tool retries.")

Built an agent with model retry + fallback and read-only tool retries.


##### 3. Protect Sensitive Information

In [72]:
from langchain.agents.middleware import PIIMiddleware

PHONE_PATTERN = r"(?:\+?\d{1,3}[\s.-]?)?(?:\(\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{4}"

pii_agent = create_agent(
    model=reliable_llm,
    tools=support_tools,
    system_prompt=SYSTEM_PROMPT,
    context_schema=AgentContext,
    middleware=[
        PIIMiddleware("email", strategy="redact"),                                  # -> [REDACTED_EMAIL]
        PIIMiddleware("credit_card", strategy="mask"),                              # -> ****-****-****-1234
        PIIMiddleware("phone_number", detector=PHONE_PATTERN, strategy="block"),    # raises
    ],
)

def ask_pii(question):
    try:
        result = pii_agent.invoke(
            {"messages": [{"role": "user", "content": question}]},
            context={"customer_id": 101},
        )
    except Exception as exc:  # PIIDetectionError for blocked content
        print("BLOCKED:", type(exc).__name__)
        return None
    print("MODEL SAW:", result["messages"][0].content)   # after redaction and masking
    print("FINAL:", result["messages"][-1].content)
    return result

# Email is redacted and the card is masked; the agent still runs.
ask_pii("Where is my latest order? Email me at asha@example.com, card 4111 1111 1111 1111.")

# A phone number is blocked before the model is ever called.
ask_pii("Call me on +919876543210 about my latest order.")

MODEL SAW: Where is my latest order? Email me at [REDACTED_EMAIL], card **** **** **** 1111.
FINAL: Your latest order is **ORD-1003** from **Burger Barn**.

- **Items:** Veg Burger x2; Fries x1
- **Status:** OUT_FOR_DELIVERY
- **Total:** ₹350.00
- **Placed:** 2026-11-06 20:14:37
- **Promised delivery by:** 2026-11-06 20:54:37

Your order is currently out for delivery and has not been delivered yet. I’ve noted your email address and card ending in 1111 for reference, but I don’t have a tool to send emails directly.
BLOCKED: PIIDetectionError


In [73]:
import logging, time
from typing import Any
from langchain.agents.middleware import AgentState, before_model, hook_config, wrap_tool_call
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.runtime import Runtime

logger = logging.getLogger("support_agent")
WRITE_TOOLS = {"request_refund"}

@wrap_tool_call
def time_tool_calls(request, handler):
    started = time.perf_counter()
    result = handler(request)                                   # run the real tool
    logger.info("tool=%s ms=%.0f", request.tool_call["name"], (time.perf_counter() - started) * 1000)
    return result

@before_model
@hook_config(can_jump_to=["end"])                               # declares that this hook may end the run
def stop_long_runs(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if len(state["messages"]) >= 50:
        handoff = AIMessage("I could not finish this. Let me hand you to a person.")
        return {"messages": [handoff], "jump_to": "end"}
    return None                                                 # nothing to change, carry on

@wrap_tool_call
def safe_tool_errors(request, handler):
    name = request.tool_call["name"]
    try:
        return handler(request)
    except psycopg.OperationalError:
        logger.exception("database failure in %s", name)        # detail stays in the logs
        body = {"error": "TEMPORARY_DATABASE_FAILURE", "retryable": name not in WRITE_TOOLS}
    except Exception:
        logger.exception("unexpected failure in %s", name)
        body = {"error": "TOOL_FAILURE", "retryable": False}
    return ToolMessage(content=json.dumps(body), tool_call_id=request.tool_call["id"], status="error")

In [74]:
from langchain.agents.middleware import SummarizationMiddleware

full_agent = create_agent(
    model=reliable_llm,
    tools=support_tools,
    system_prompt=SYSTEM_PROMPT,
    context_schema=AgentContext,
    middleware=[
        time_tool_calls,                                                                   # 1  yours
        safe_tool_errors,                                                                  # 2  yours
        ModelCallLimitMiddleware(run_limit=8, exit_behavior="end"),                        # 3
        PIIMiddleware("email", strategy="redact"),                                         # 4
        SummarizationMiddleware(model=reliable_llm, trigger=("tokens", 4000),
                                keep=("messages", 20)),                                    # 5
        ModelFallbackMiddleware(backup_llm),                                               # 6
        ModelRetryMiddleware(max_retries=2, on_failure="error"),                           # 7
        ToolRetryMiddleware(max_retries=2, tools=["get_order_details", "get_latest_order"],
                            retry_on=(psycopg.OperationalError, TimeoutError),
                            on_failure="error"),                                           # 8  reads only
        ToolCallLimitMiddleware(tool_name="get_order_details", run_limit=3),               # 9
    ],
)
print("Built the full middleware stack.")

Built the full middleware stack.


In [75]:
# --- A logging callback: one short line per model call and tool call, request and response included.
import time
from langchain_core.callbacks import BaseCallbackHandler

def _short(value, limit=200):
    text = str(value).replace("\n", " ")
    return text if len(text) <= limit else text[:limit] + "..."

class StepLogger(BaseCallbackHandler):
    """Prints one short line per model call and tool call, with request and response data."""

    def __init__(self):
        self.started = {}

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        self.started[run_id] = time.perf_counter()
        last = messages[0][-1]
        print(f"[model  >] {len(messages[0])} messages in, last {last.type}: {_short(last.content, 120)}")

    def on_llm_end(self, response, *, run_id, **kwargs):
        ms = (time.perf_counter() - self.started.pop(run_id, time.perf_counter())) * 1000
        message = response.generations[0][0].message
        if message.tool_calls:
            calls = [(c["name"], c["args"]) for c in message.tool_calls]
            print(f"[model  <] {ms:.0f} ms, asks for tools: {_short(calls)}")
        else:
            print(f"[model  <] {ms:.0f} ms, answer: {_short(message.content)}")

    def on_tool_start(self, serialized, input_str, *, run_id, inputs=None, **kwargs):
        self.started[run_id] = time.perf_counter()
        print(f"[tool   >] {serialized.get('name')} args={_short(inputs or input_str)}")

    def on_tool_end(self, output, *, run_id, **kwargs):
        ms = (time.perf_counter() - self.started.pop(run_id, time.perf_counter())) * 1000
        print(f"[tool   <] {ms:.0f} ms, result: {_short(getattr(output, 'content', output))}")

    def on_tool_error(self, error, *, run_id, **kwargs):
        self.started.pop(run_id, None)
        print(f"[tool   !] {type(error).__name__}: {_short(error)}")

    def on_llm_error(self, error, *, run_id, **kwargs):
        self.started.pop(run_id, None)
        print(f"[model  !] {type(error).__name__}: {_short(error)}")

step_logger = StepLogger()   # passed as callbacks=[step_logger] in the run config

In [76]:
def ask_full(question, customer_id=101):
    result = full_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"callbacks": [step_logger]},
        context={"customer_id": customer_id},
    )
    print("FINAL:", result["messages"][-1].content)
    return result

ask_full("Where is my latest order?")

[model  >] 2 messages in, last human: Where is my latest order?
[model  <] 672 ms, asks for tools: [('get_latest_order', {})]
[tool   >] get_latest_order args={}
[tool   <] 18 ms, result: {"ok": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status": "OUT_FOR_DELIVERY", "total_amount": 350.0, "created_at": "2026-11-06 20:14:37....
[model  >] 4 messages in, last tool: {"ok": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status"...
[model  <] 1613 ms, answer: Your latest order is **out for delivery**.  - **Order ID:** ORD-1003 - **Restaurant:** Burger Barn - **Items:** Veg Burger x2; Fries x1 - **Status:** OUT_FOR_DELIVERY - **Total:** ₹350.00 - **Promised...
FINAL: Your latest order is **out for delivery**.

- **Order ID:** ORD-1003
- **Restaurant:** Burger Barn
- **Items:** Veg Burger x2; Fries x1
- **Status:** OUT_FOR_DELIVERY
- **Total:** ₹350.00
- **Promised deliver

{'messages': [HumanMessage(content='Where is my latest order?', additional_kwargs={}, response_metadata={}, id='c50c9a35-1b50-41cc-8d25-2c00c65df808'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 680, 'total_tokens': 727, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 30, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3.5-super-vl:free', 'system_fingerprint': None, 'id': 'gen-1794028653-5TDOUVL594LOL3cXohoy', 'finish_reason': 'tool_calls', 'logp

In [77]:
ask_full("My email is asha@example.com. Where is my latest order?")

[model  >] 2 messages in, last human: My email is [REDACTED_EMAIL]. Where is my latest order?
[model  <] 805 ms, asks for tools: [('get_latest_order', {})]
[tool   >] get_latest_order args={}
[tool   <] 18 ms, result: {"ok": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status": "OUT_FOR_DELIVERY", "total_amount": 350.0, "created_at": "2026-11-06 20:14:37....
[model  >] 4 messages in, last tool: {"ok": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status"...
[model  <] 1746 ms, answer: Your latest order is **ORD-1003** from **Burger Barn** (Veg Burger x2, Fries x1), totaling ₹350.00.  **Status:** Out for delivery   **Promised delivery time:** 2026-11-06 20:54:37 (UTC)  It hasn't bee...
FINAL: Your latest order is **ORD-1003** from **Burger Barn** (Veg Burger x2, Fries x1), totaling ₹350.00.

**Status:** Out for delivery  
**Promised delivery time:** 2026-11-06 20:54:3

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL]. Where is my latest order?', additional_kwargs={}, response_metadata={}, id='49cfafe2-5c02-4533-942a-ac51ac21d88e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 690, 'total_tokens': 758, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 51, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3.5-super-vl:free', 'system_fingerprint': None, 'id': 'gen-1794028655-FQIFiOaEoYJtISxL4NXt', 'finis

In [78]:
import logging, time
from typing import Any
from langchain.agents.middleware import AgentState, before_model, hook_config, wrap_tool_call
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.runtime import Runtime

logger = logging.getLogger("support_agent")
WRITE_TOOLS = {"request_refund"}

@wrap_tool_call
def time_tool_calls(request, handler):
    started = time.perf_counter()
    result = handler(request)                                   # run the real tool
    logger.info("tool=%s ms=%.0f", request.tool_call["name"], (time.perf_counter() - started) * 1000)
    return result

@before_model
@hook_config(can_jump_to=["end"])                               # declares that this hook may end the run
def stop_long_runs(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if len(state["messages"]) >= 50:
        handoff = AIMessage("I could not finish this. Let me hand you to a person.")
        return {"messages": [handoff], "jump_to": "end"}
    return None                                                 # nothing to change, carry on

@wrap_tool_call
def safe_tool_errors(request, handler):
    name = request.tool_call["name"]
    try:
        return handler(request)
    except psycopg.OperationalError:
        logger.exception("database failure in %s", name)        # detail stays in the logs
        body = {"error": "TEMPORARY_DATABASE_FAILURE", "retryable": name not in WRITE_TOOLS}
    except Exception:
        logger.exception("unexpected failure in %s", name)
        body = {"error": "TOOL_FAILURE", "retryable": False}
    return ToolMessage(content=json.dumps(body), tool_call_id=request.tool_call["id"], status="error")        

In [82]:
from langchain.agents.middleware import SummarizationMiddleware

full_agent = create_agent(
    model=reliable_llm,
    tools=support_tools,
    system_prompt=SYSTEM_PROMPT,
    context_schema=AgentContext,
    middleware=[
        time_tool_calls,                                                                   # 1  yours
        safe_tool_errors,                                                                  # 2  yours
        ModelCallLimitMiddleware(run_limit=8, exit_behavior="end"),                        # 3
        PIIMiddleware("email", strategy="redact"),                                         # 4
        SummarizationMiddleware(model=reliable_llm, trigger=("tokens", 4000),
                                keep=("messages", 20)),                                    # 5
        ModelFallbackMiddleware(backup_llm),                                               # 6
        ModelRetryMiddleware(max_retries=2, on_failure="error"),                           # 7
        ToolRetryMiddleware(max_retries=2, tools=["get_order_details", "get_latest_order"],
                            retry_on=(psycopg.OperationalError, TimeoutError),
                            on_failure="error"),                                           # 8  reads only
        ToolCallLimitMiddleware(tool_name="get_order_details", run_limit=3),               # 9
    ],
)
print("Built the full middleware stack.")   


Built the full middleware stack.


In [83]:
# --- A logging callback: one short line per model call and tool call, request and response included.
import time
from langchain_core.callbacks import BaseCallbackHandler


def _short(value, limit=200):
    text = str(value).replace("\n", " ")
    return text if len(text) <= limit else text[:limit] + "..."

class StepLogger(BaseCallbackHandler):
    """Prints one short line per model call and tool call, with request and response data."""

    def __init__(self):
        self.started = {}

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        self.started[run_id] = time.perf_counter()
        last = messages[0][-1]
        print(f"[model  >] {len(messages[0])} messages in, last {last.type}: {_short(last.content, 120)}")

    def on_llm_end(self, response, *, run_id, **kwargs):
        ms = (time.perf_counter() - self.started.pop(run_id, time.perf_counter())) * 1000
        message = response.generations[0][0].message
        if message.tool_calls:
            calls = [(c["name"], c["args"]) for c in message.tool_calls]
            print(f"[model  <] {ms:.0f} ms, asks for tools: {_short(calls)}")
        else:
            print(f"[model  <] {ms:.0f} ms, answer: {_short(message.content)}")

    def on_tool_start(self, serialized, input_str, *, run_id, inputs=None, **kwargs):
        self.started[run_id] = time.perf_counter()
        print(f"[tool   >] {serialized.get('name')} args={_short(inputs or input_str)}")

    def on_tool_end(self, output, *, run_id, **kwargs):
        ms = (time.perf_counter() - self.started.pop(run_id, time.perf_counter())) * 1000
        print(f"[tool   <] {ms:.0f} ms, result: {_short(getattr(output, 'content', output))}")

    def on_tool_error(self, error, *, run_id, **kwargs):
        self.started.pop(run_id, None)
        print(f"[tool   !] {type(error).__name__}: {_short(error)}")

    def on_llm_error(self, error, *, run_id, **kwargs):
        self.started.pop(run_id, None)
        print(f"[model  !] {type(error).__name__}: {_short(error)}")

step_logger = StepLogger()   # passed as callbacks=[step_logger] in the run config



In [86]:
def ask_full(question, customer_id=101):
    result = full_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"callbacks": [step_logger]},
        context={"customer_id": customer_id},
    )
    if result.get("status_code") and result["status_code"] >= 400:
        print("ERROR:", result["status_code"], result.get("error", result.get("message")))
    for message in result["messages"]:
        # The call is requested on the AIMessage; the ToolMessage only carries the result.
        for call in getattr(message, "tool_calls", None) or []:
            print("TOOL CALL:", call["name"], str(call["args"])[:400])
        if message.type == "tool" and message.content:
            print("TOOL RESULT:", str(message.content)[:400])
    print("FINAL:", result["messages"][-1].content)
    return result

ask_full("Where is my latest order?")

[model  >] 2 messages in, last human: Where is my latest order?
[model  <] 968 ms, asks for tools: [('get_latest_order', {})]
[tool   >] get_latest_order args={}
[tool   <] 18 ms, result: {"ok": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status": "OUT_FOR_DELIVERY", "total_amount": 350.0, "created_at": "2026-11-06 20:14:37....
[model  >] 4 messages in, last tool: {"ok": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status"...
[model  <] 1614 ms, answer: Your latest order (ORD-1003) from **Burger Barn** is currently **out for delivery**. It was placed on 2026-11-06 at 20:14 and is expected by 20:54 today.
TOOL CALL: get_latest_order {}
TOOL RESULT: {"ok": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status": "OUT_FOR_DELIVERY", "total_amount": 350.0, "created_at": "2026-11-06 20:14:37.166546+00:00", "promi

{'messages': [HumanMessage(content='Where is my latest order?', additional_kwargs={}, response_metadata={}, id='4797242b-d11c-4db7-a31b-d281e14c8a18'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 680, 'total_tokens': 727, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 30, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3.5-super-vl:free', 'system_fingerprint': None, 'id': 'gen-1794030781-JdHwXeoqtYadIF5fZt2G', 'finish_reason': 'tool_calls', 'logp

In [87]:
ask_full("my email is asha@example.com. Where is my latest order?")

[model  >] 2 messages in, last human: my email is [REDACTED_EMAIL]. Where is my latest order?
[model  <] 2259 ms, asks for tools: [('get_latest_order', {})]
[tool   >] get_latest_order args={}
[tool   <] 10 ms, result: {"ok": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status": "OUT_FOR_DELIVERY", "total_amount": 350.0, "created_at": "2026-11-06 20:14:37....
[model  >] 4 messages in, last tool: {"ok": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status"...
[model  <] 2073 ms, answer: Your latest order is **ORD-1003** from **Burger Barn** (Veg Burger x2, Fries x1), totaling ₹350.00.  **Status:** Out for delivery   **Promised delivery by:** 2026-11-06 20:54 (IST)  It hasn't been del...
TOOL CALL: get_latest_order {}
TOOL RESULT: {"ok": true, "order": {"order_id": "ORD-1003", "restaurant": "Burger Barn", "items": "Veg Burger x2; Fries x1", "status": "OUT_FOR_DELIVERY"

{'messages': [HumanMessage(content='my email is [REDACTED_EMAIL]. Where is my latest order?', additional_kwargs={}, response_metadata={}, id='987f72ae-f04d-48da-af8c-7e952744b7fd'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 690, 'total_tokens': 748, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 41, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3.5-super-vl:free', 'system_fingerprint': None, 'id': 'gen-1794030969-W2VcE4bAMKY46Hp3urrJ', 'finis